**Effective ownership:** how much an entity really owns of a company once you follow the whole chain of owners in between.

Example: A owns 50% of B, and B owns 40% of C. A doesn't own C directly, but through B it effectively owns **50% × 40% = 20%** of C. To get this, walk the chain from A to B to C and multiply the percentages along the way.

If A also reaches C by another route (A owns 100% of D, and D owns 10% of C), add the routes: 20% + 10% = **30%**.

In [0]:
CREATE OR REPLACE  TABLE pep_dev.eval.edges_pct
USING DELTA
AS
SELECT owner_id,asset_id ,owner_type,asset_type, pct_lo/100 AS share FROM pep_dev.eval.edges_current
WHERE pct_lo IS NOT NULL

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.paths
USING DELTA 
AS
SELECT owner_id as  top_id,owner_type as top_type, asset_id, share, 1 AS depth, array(owner_id, asset_id) AS path 
FROM pep_dev.eval.edges_pct;

In [0]:
SELECT count(*), min(depth), max(depth) FROM pep_dev.eval.paths

In [0]:
    %python

    for r in range(2,31):
        res=spark.sql(""" 
                INSERT INTO pep_dev.eval.paths
                SELECT 
                p.top_id , 
                p.top_type, 
                e.asset_id,
                p.share*e.share,
                
                :r,
                array_append(p.path,e.asset_id)
                
                

                FROM pep_dev.eval.paths as p
                JOIN pep_dev.eval.edges_pct as e
                ON p.asset_id = e.owner_id
            
                WHERE p.depth= :r-1 AND 
                NOT array_contains(p.path,e.asset_id)

                
                """, args={"r":r})
        n=res.collect()[0]["num_inserted_rows"]
        print(f"depth {r}: {n} new paths")
        if n == 0:
            break


In [0]:
SELECT x.pos, x.id, le.name, le.country
FROM (
  SELECT posexplode(path) AS (pos, id)
  FROM (SELECT path FROM pep_dev.eval.paths
        WHERE depth = 12 AND share = 1 LIMIT 1)
) AS x
LEFT JOIN pep_dev.silver.legal_entity AS le ON le.canonical_id = x.id
ORDER BY x.pos